# Stage 2: ランドマーク検出器 (HRNet-W32) 訓練

**実行環境**: Google Colab（GPU 必須）

## 手順
1. Google Drive をマウントし、リポジトリと `train/dataset/phase2/` を配置
2. 設定セルを編集して実行
3. 訓練済み weights を `train/runs/phase2_best.pt` に保存

## Stage 2 の入力
各椎体/領域の BBoxクロップ画像 (256×256) → 4ch heatmap 出力。
null ランドマーク（未アノテーション）はロスから自動除外されます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

# ---- 設定 (ここを編集) ----
REPO_ROOT   = Path("/content/drive/MyDrive/anglist")
DATA_DIR    = REPO_ROOT / "train/dataset/phase2"
RUNS_DIR    = REPO_ROOT / "train/runs"
BACKBONE    = "hrnet_w32"  # または "smallunet"（テスト用）
SIGMA       = 5.0
EPOCHS      = 100
BATCH_SIZE  = 8
LR          = 1e-4
AUGMENT     = True
REGION_FILTER = None  # None=全領域 / ["L3","L4"] のように絞れる
# ---------------------------

sys.path.insert(0, str(REPO_ROOT))
import os; os.makedirs(str(RUNS_DIR), exist_ok=True)

In [ ]:
!pip install timm albumentations segmentation-models-pytorch -q

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from train.dataset_phase2 import Phase2Dataset
from train.model_hrnet import get_stage2_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

full_ds = Phase2Dataset(
    str(DATA_DIR),
    mode="stage2",
    sigma=SIGMA,
    augment=AUGMENT,
    region_filter=REGION_FILTER,
)
n_val = max(1, int(len(full_ds) * 0.1))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Train: {n_train}, Val: {n_val}")

model = get_stage2_model(backbone=BACKBONE, num_channels=4, pretrained=True).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
import torch.nn.functional as F

def masked_mse(pred, target, valid_mask):
    """valid_mask=True のチャネルのみ MSE を計算する。"""
    # pred, target: (B, 4, H, W)  valid_mask: (B, 4)
    diff = (pred - target) ** 2  # (B, 4, H, W)
    mask = valid_mask.float().unsqueeze(-1).unsqueeze(-1)  # (B, 4, 1, 1)
    loss = (diff * mask).sum() / mask.sum().clamp(min=1.0)
    return loss

best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    # ---- 訓練 ----
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        img    = batch["image"].to(device)
        hm     = batch["heatmap"].to(device)
        vmask  = batch["valid_mask"].to(device)
        if not vmask.any():
            continue
        pred = model(img)
        loss = masked_mse(pred, hm, vmask)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()

    # ---- 検証 ----
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            img   = batch["image"].to(device)
            hm    = batch["heatmap"].to(device)
            vmask = batch["valid_mask"].to(device)
            pred  = model(img)
            val_loss += masked_mse(pred, hm, vmask).item()

    n_tb = max(1, len(train_loader))
    n_vb = max(1, len(val_loader))
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}  train={train_loss/n_tb:.4f}  val={val_loss/n_vb:.4f}")

    if val_loss / n_vb < best_val_loss:
        best_val_loss = val_loss / n_vb
        torch.save(model.state_dict(), str(RUNS_DIR / "phase2_best.pt"))

print(f"Best val loss: {best_val_loss:.4f}  → {RUNS_DIR / 'phase2_best.pt'}")

In [ ]:
# ONNX エクスポート
!python {REPO_ROOT}/train/export_onnx_phase2.py \
    --checkpoint {RUNS_DIR}/phase2_best.pt \
    --output     {RUNS_DIR}/phase2_best.onnx \
    --backbone   {BACKBONE}